In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('data/tagged_reviews.csv')
df.head()

,source,text,rating,version,date,title,storefront,author,review_id,theme_tags
0,app_store,"PLEASE bring back the old smaller pictures, th...",1,9.3.1,NaN,NEW UI SUCKS,us,wsonja112,0,"{""feature requests"": ""negative""}"
1,app_store,The UI is clunky and requiring invites to use ...,1,9.3.1,NaN,Poor design and scammy behavior,us,scknight4,1,"{""forced invite wall"": ""negative"", ""performanc..."
2,app_store,It is good at what it does and it’s free nd no...,5,9.3.1,NaN,Happy,us,Gstarbanana,2,"{""positive overall experience"": ""positive""}"
3,app_store,I have to share that I am drifting away from t...,2,9.3.1,NaN,Worse than Google Maps,us,brenda y. g.,3,"{""ranking and rating mechanics"": ""negative"", ""..."
4,app_store,All I want to do is find a restaurant near my ...,1,9.3.0,NaN,Useless,us,DestructoTex,4,"{""forced invite wall"": ""negative"", ""map and na..."


In [6]:
"""
Reads data/tagged_reviews.csv (the LLM-tagged corpus) and turns the per-review
theme/sentiment JSON into the analytical centerpiece:
  - how OFTEN each theme appears (frequency)
  - how NEGATIVE it is when it appears (severity)
  - a frequency x severity ranking of pain points
"""

import json
import pandas as pd


def main():
    df = pd.read_csv("data/tagged_reviews.csv")

    long_rows = []
    for _, row in df.iterrows():
        try:
            tags = json.loads(row["theme_tags"])
        except (json.JSONDecodeError, TypeError):
            continue
        if not isinstance(tags, dict):
            continue
        for theme, sentiment in tags.items():
            if theme.startswith("_"):      
                continue
            long_rows.append({
                "review_id": row.get("review_id"),
                "theme": theme,
                "sentiment": sentiment,
            })

    long = pd.DataFrame(long_rows)
    total_reviews = len(df)
    print(f"Total reviews: {total_reviews}")
    print(f"Total theme mentions: {len(long)}\n")


    # FREQUENCY: how many reviews mention each theme, and what share
    #    of all reviews that is.
    freq = (long.groupby("theme")["review_id"].nunique()
                .sort_values(ascending=False)
                .rename("n_reviews"))
    freq_pct = (freq / total_reviews * 100).round(1).rename("pct_of_reviews")


    # SEVERITY: of the mentions of each theme, what share are negative.
    # Higher = more painful. (Also track positive/neutral for context.)
    sent_counts = (long.groupby(["theme", "sentiment"]).size()
                       .unstack(fill_value=0))
    for col in ["negative", "neutral", "positive"]:
        if col not in sent_counts.columns:
            sent_counts[col] = 0
    sent_counts["total"] = sent_counts[["negative", "neutral", "positive"]].sum(axis=1)
    sent_counts["pct_negative"] = (sent_counts["negative"] /
                                   sent_counts["total"] * 100).round(1)


    #  COMBINE into the frequency x severity table, ranked by a simple
    #    priority score = (# reviews mentioning) x (% negative).
    #    Frequent AND negative themes rise to the top.
    matrix = pd.DataFrame({
        "n_reviews": freq,
        "pct_of_reviews": freq_pct,
        "pct_negative": sent_counts["pct_negative"],
        "n_negative": sent_counts["negative"],
        "n_positive": sent_counts["positive"],
    })
    matrix["priority_score"] = (matrix["n_reviews"] *
                                matrix["pct_negative"] / 100).round(1)
    matrix = matrix.sort_values("priority_score", ascending=False)

    print("FREQUENCY x SEVERITY MATRIX (ranked by priority)\n")
    print(matrix.to_string())

    matrix.to_csv("data/theme_priority_matrix.csv")

    top = matrix.index[0]
    print(f"\nTop pain point: '{top}' — mentioned in {int(matrix.loc[top,'n_reviews'])} "
          f"reviews ({matrix.loc[top,'pct_of_reviews']}%), "
          f"{matrix.loc[top,'pct_negative']}% negative.")


if __name__ == "__main__":
    main()

Total reviews: 825
Total theme mentions: 1566

FREQUENCY x SEVERITY MATRIX (ranked by priority)

                              n_reviews  pct_of_reviews  pct_negative  n_negative  n_positive  priority_score
theme                                                                                                        
feature requests                    309            37.5          65.3         226         106           201.8
forced invite wall                  178            21.6          97.3         214           4           173.2
ranking and rating mechanics        219            26.5          56.3         130          98           123.3
performance and clunkiness          103            12.5          95.3         101           4            98.2
data privacy and account             77             9.3          93.9          77           4            72.3
map and navigation                  104            12.6          64.8          68          36            67.4
positive overall experi

## Commentary

From the analysis results, feature requests is the most-mentioned theme, accounting for 37.5% of total reviews, 65.3% of which carry negative sentiment. In this notebook, I'll further explore which specific features are most requested. The forced invite wall also appears frequently, occupying 21.6% of reviews, with 97.3% of those carrying negative sentiment. Despite its lower priority score than feature requests, I argue the forced invite wall is the more important theme to investigate, because it is a single, well-defined feature, whereas feature requests are diffuse—more time-consuming to analyze and harder to weigh trade-offs across when deciding what to update. The invite wall's 97.3% negativity is near-unanimous and concentrated on one fixable surface, which is a stronger action signal than a slightly higher priority score spread across a vague bucket.


I also want to state a key limitation. The mean rating of users who left a written review is notably lower than Beli's overall rating on both the App Store and Google Play. This reflects voluntary response bias—a form of self-selection where users who opt to leave reviews skew toward stronger, often more negative, opinions than the silent majority. As a result, I can't claim these proportions reflect how all Beli users feel; the sample over-represents dissatisfied users.


Crucially, this limitation does not undercut the comparison driving my recommendation. The bias inflates negativity across every theme equally, since all themes are drawn from the same review pool—so it shifts absolute sentiment levels but not the ranking between themes. Even within this negatively-skewed sample, the invite wall's 97.3% negativity stands out sharply against other themes, including feature requests at 65.3%. So while I can't say what share of all users dislike the invite wall, I can say that among users who voice an opinion, opposition to it is near-unanimous and far more concentrated than for any other theme—which is what makes it actionable.